In [0]:
from pyspark.sql.functions import col, to_timestamp, get_json_object, from_json, explode_outer, lit
from pyspark.sql.types import ArrayType, StructType, StructField, StringType

BRONZE_TABLE = "gharchive_dev.v1_pyspark.gharchive_bronze"
SILVER_TABLE = "gharchive_dev.v1_pyspark.gharchive_silver"

# NOTE: Reads the ENTIRE bronze table — full scan, full recompute
# (V2/V3 DLT pipelines handle this incrementally)
df_bronze = spark.read.table(BRONZE_TABLE)

# Flatten repo struct
df = df_bronze
if "repo" in df.columns:
    df = (df
        .withColumn("repo_id", col("repo.id"))
        .withColumn("repo_name", col("repo.name"))
        .withColumn("repo_url", col("repo.url"))
    )

# Flatten actor struct
if "actor" in df.columns:
    df = (df
        .withColumn("actor_id", col("actor.id"))
        .withColumn("actor_login", col("actor.login"))
        .withColumn("actor_display_login", col("actor.display_login"))
        .withColumn("actor_avatar_url", col("actor.avatar_url"))
    )

# Extract commit fields from payload JSON string
# payload is stored as raw JSON string in bronze to avoid schema merge conflicts
COMMIT_SCHEMA = ArrayType(StructType([
    StructField("sha", StringType()),
    StructField("message", StringType()),
    StructField("author", StructType([
        StructField("name", StringType()),
        StructField("email", StringType()),
    ])),
]))

commits_json = get_json_object(col("payload"), "$.commits")
df = df.withColumn("_commits", from_json(commits_json, COMMIT_SCHEMA))

# Explode commits (only PushEvents have them, others will be null)
df = df.withColumn("commit", explode_outer("_commits"))
df = (df
    .withColumn("commit_sha", col("commit.sha"))
    .withColumn("commit_message", col("commit.message"))
    .withColumn("commit_author_name", col("commit.author.name"))
    .withColumn("commit_author_email", col("commit.author.email"))
)
df = df.drop("commit", "_commits")

# Extract action from payload (common across event types)
df = df.withColumn("payload_action", get_json_object(col("payload"), "$.action"))

# Parse created_at to timestamp
df = df.withColumn("created_at", to_timestamp("created_at"))

# Drop original nested columns
df_silver = df.drop("repo", "actor", "payload", "org", "_rescued_data")

# Write as Delta table (full overwrite each run)
df_silver.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)

print(f"Silver table written: {SILVER_TABLE}")
print(f"Row count: {spark.read.table(SILVER_TABLE).count()}")